# EEG tutorial BIDS

**Learning goals**
- Get familiar with BIDS;
- Get familiar with MNE-BIDS to make a dataset BIDS compliant.

**Summary**

This notebook shows the basics of the Brain Imaging Data Structure (BIDS) and how to use BIDS-MNE to make an arbitrary dataset BIDS compliant. This notebook uses real EEG data shared on Zenodo. 

**Content**

* Step 1: Downloading and setting up EEG data
* Step 2: Making the dataset BIDS compliant


In [1]:
import yaml
import datetime
import json
import os
import glob
from pprint import pprint
import warnings

import mne
from mne_bids import BIDSPath, make_dataset_description, write_raw_bids
import numpy as np

warnings.simplefilter(action="ignore", category=RuntimeWarning)
warnings.simplefilter(action="ignore", category=FutureWarning)


## Step 1: Downloading and setting up EEG data
The dataset contains data of three subjects. For each subject, data of a single session is available. Within a session, a resting state recording with eyes open and eyes closed has been performed at the beginning and the end, which you can ignore. In the main part of the session, an auditory oddball paradigm was executed. For two different stimulus onset asynchrony (SOA) values, a fast and a slow condition, 80 trials of oddball ERP data were collected. Within a trial, you should find 90 stimuli (75 low-pitched non-targets and 15 high-pitched targets). Please note, that the actual SOA values vary between subjects.

In addition to EEG channels, a number of other non-EEG channels have been recorded.

1. Retrieve the data from [Zenodo](https://zenodo.org/record/4066633) and store it at a convenient location locally;
1. In the code snippet below, set the `CONFIG_FILE_PATH` to point the folder where the `config.yaml` file is located;
1. Open the `config.yaml` file in a text editor, and change the `data_path` on line 1 to point to the path where you downloaded the data.

In [2]:
# Adjust this CONFIG_FILE_PATH to point to the config.yaml file
CONFIG_FILE_PATH = os.path.join(os.getcwd(), "config.yaml")

# Load the file
conf = yaml.load(open(CONFIG_FILE_PATH, "r"), Loader=yaml.FullLoader)


## Step 2: Making the dataset BIDS compliant

The code below performs a basic BIDS restructuring and anonymization, including:
- Reading the raw data
- Setting EEG information such as the line noise frequency and amplifier manufacturer
- Setting participant information and anonymizing the dataset, including setting the measurement data to a default value, the participant id, sex, handedness, birthday (anonimyzed), weight and height.
- Setting the electrode montage
- Makeing sure all markers are in place
- Saving to a normalized BIDS file structure


In [3]:
non_eeg_channels = ["EOGvu", "x_EMGl", "x_GSR", "x_Respi", "x_Pulse", "x_Optic"]
line_freq = 50.0

# Note, these are mock-up values, as they were not present in the dataset
sex = ["F", "M", "F"]  # F/M
age = [20, 21, 22]  # years
hand = ["L", "R", "R"] # R/L/A
weight = [70, 71, 72]  # kg
height = [1.70, 1.75, 1.80]  # m

for i_session, session in enumerate(conf["auditory_oddball"]["session_list"]):
    for condition in conf["auditory_oddball"]["condition_list"]:
        
        # Find files
        search_path = os.path.join(conf['data_path'], session, f"*{condition}.vhdr")
        eeg_filepaths = sorted(glob.glob(search_path))

        if len(eeg_filepaths) == 0:
            print(f"No files found for session {session} and condition {condition}!")
            break

        for i_run, eeg_filepath in enumerate(eeg_filepaths):
            print(f"Loading file {os.path.basename(eeg_filepath)}")

            # Read raw continuous data
            raw = mne.io.read_raw_brainvision(eeg_filepath, misc=non_eeg_channels, verbose=False)

            # Select EEG channels only
            raw.pick("eeg", verbose=False)

            # Add EEG info
            raw.info["line_freq"] = line_freq
            raw.info["device_info"] = {
                "type": "EEG",
                "model": "BrainAmp DC"
            }
    
            # Add anonymized participant info
            raw.info.set_meas_date(datetime.datetime(year=1900, month=1, day=1, tzinfo=datetime.timezone.utc))
            raw.info["subject_info"] = {
                "id": 1+i_session,
                "sex": 1 if sex[i_session] == "M" else 2,  # Subject sex (0=unknown, 1=male, 2=female).
                "hand": 1 if hand[i_session] == "R" else 2,  # Handedness (1=right, 2=left, 3=ambidextrous).
                "birthday": datetime.datetime(year=1900 - age[i_session], month=1, day=1, tzinfo=datetime.timezone.utc),
                "weight": weight[i_session],
                "height": height[i_session],
            }
    
            # Set electrode positions
            montage = mne.channels.make_standard_montage("standard_1020")
            raw.set_montage(montage, verbose=False)
    
            # Required because annotations contains an orig_time which relates to the meas_time which we changed (anonymized)
            annot = mne.Annotations(raw.annotations.onset, raw.annotations.duration, raw.annotations.description)
            annot.delete([i for i, val in enumerate(annot.description) if val in ["BAD boundary", "EDGE boundary"]])
            raw.set_annotations(annot)
            
            # Set path
            bids_path = BIDSPath(
                subject=f"{1+i_session:02d}",
                session="01",
                task=condition,
                run=1 + i_run,
                datatype="eeg",
                root=os.path.join(conf["data_path"], "bids"),
            )
            
            # Save files
            write_raw_bids(
                raw=raw,
                bids_path=bids_path,
                overwrite=True,
                allow_preload=True,
                format="BrainVision",
                montage=montage,
                verbose=False,
            )

print("Done!")


Loading file Oddball_Run_2_Trial_016_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_017_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_018_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_019_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_020_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_036_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_037_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_038_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_039_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_040_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_056_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_057_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_058_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_059_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_060_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_076_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Trial_077_SOA_0.208_fast.vhdr
Loading file Oddball_Run_2_Tria

## Step 3: Further exercises
The above cells provide a starting point for making a dataset BIDs compatible. 
- As an exercise, try changing the EEG tutorial ERP to using this BIDS compliant dataset. You might realize how easy it is to read and use a dataset that is organized according to the BIDS specification. That is, make it a habit to make your data BIDS compatible as soon as you get the data in.
- Also [MOABB](https://neurotechx.github.io/moabb/) has a BIDS compatible data loader, see [here](https://github.com/NeuroTechX/moabb/pull/724), which makes it really easy to get your BIDS dataset integrated in MOABB.


## Step 4: Further reading
The above cells provide a starting point for making a dataset BIDs compatible. This relied on the BIDS specification, of which information can be found [here](https://bids.neuroimaging.io/). Documentation about the EEG BIDS extension can be found [here](https://bids-specification.readthedocs.io/en/stable/modality-specific-files/electroencephalography.html). Specifically, publication associated to BIDS are:

- Gorgolewski, K.J., Auer, T., Calhoun, V.D., Craddock, R.C., Das, S., Duff, E.P., Flandin, G., Ghosh, S.S., Glatard, T., Halchenko, Y.O., Handwerker, D.A., Hanke, M., Keator, D., Li, X., Michael, Z., Maumet, C., Nichols, B.N., Nichols, T.E., Pellman, J., Poline, J.-B., Rokem, A., Schaefer, G., Sochat, V., Triplett, W., Turner, J.A., Varoquaux, G., Poldrack, R.A. (2016). The brain imaging data structure, a format for organizing and describing outputs of neuroimaging experiments. Scientific Data, 3 (160044). doi:[10.1038/sdata.2016.44](https://doi.org/10.1038/sdata.2016.44)

- Pernet, C. R., Appelhoff, S., Gorgolewski, K.J., Flandin, G., Phillips, C., Delorme, A., Oostenveld, R. (2019). EEG-BIDS, an extension to the brain imaging data structure for electroencephalography. Scientific data, 6 (103). doi:[10.1038/s41597-019-0104-8](https://doi.org/10.1038/s41597-019-0104-8)

More documentation around using MNE-BIDS can be found [here](https://mne.tools/mne-bids/stable/index.html) and [here](https://mne.tools/mne-bids-pipeline/stable/).